# NFTs — In Which Titus Is Minted, And His Sinuses Remain Off-Chain

Right. The monkey pictures.

An NFT stands for **non-fungible token**. It is a digital token with its own ID and a recorded owner. It can be linked to something, such as a picture. The blockchain records who owns the token; the picture can still be copied.

Imagine Titus takes a photo of himself winning a hackathon. He creates a token linked to that photo. The record might look like this:

- Token ID: 1
- Owner: Titus
- Picture: a link to the hackathon photo

Creating that token record is called **minting**. If Titus transfers the token to Bob, the owner changes from Titus to Bob. The token ID stays 1.

If Jacob saves a copy of the photo, he gets the picture. That does not change the token's owner. Copying a picture and transferring its token are two separate actions.

Compare this with notebook 8: Alice's balance of 750 TTC tells us **how many** tokens she has. An NFT record tells us **which particular token** someone owns. We will unpack the word “non-fungible” in the next section.


We will create a token, copy the picture, try to transfer it without permission, and sell it. Then we will see what happens when royalties are skipped or the website storing the picture disappears.


In [1]:
import hashlib
import random
import statistics
from dataclasses import dataclass
from decimal import Decimal

from blockchain_lib.contracts import SmartContract, submit_call
from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Blockchain, Validator

# Previously taught: notebook 2's chain, notebook 5's waiting room,
# and notebook 6's contracts-at-addresses. New NFT types are defined below.

## 1. Fungible means you care how much. Non-fungible means you care which one.

If I borrow 10 TTC, you want 10 TTC back. You do not demand the exact ten token atoms you lent me, with their childhood memories intact.

If I borrow your phone, returning an identical model is a rather different conversation. Your photos, scratches, and appalling battery health are part of the identity.

| Ledger | Typical state | Question it answers |
| --- | --- | --- |
| Fungible token | `balances["Titus"] = 1000` | How many units does Titus hold? |
| NFT collection | `owners[1] = "Titus"` | Who holds this particular token? |

An **NFT** is a non-fungible token. The individually tracked token can refer to an artwork, ticket, domain, game item, or other claim. What that claim means depends on the surrounding system.

A digital file is still copyable. The token gives us a separately tracked registry entry. Imagine a public coat-check ticket: photocopying the coat does not change who holds ticket #1. Whether anyone honours the ticket is a second question, usually answered by a tired person at a desk.


## 2. Meet the paperwork before meeting the expensive paperwork

`TokenMetadata` describes what our token points at: a title, description, file hash, and URI. A **URI** is the location string used to retrieve it. The hash lets us compare retrieved bytes with the original commitment.

Our two exceptions make the failure cases readable: you either asked about a token that does not exist, or tried to move a token you do not own. The contract is having a bad day, but at least it can explain why.

This model stores metadata alongside the ownership mapping. In many real ERC-721 designs, `tokenURI` points to a metadata document which then points to the image. Other designs store art on-chain. A separate SHA-256 file field is **our teaching choice**, rather than a universal requirement. [ERC-721 specification](https://eips.ethereum.org/EIPS/eip-721).


In [2]:
@dataclass(frozen=True)
class TokenMetadata:
    """Describe the off-chain artifact without storing its file bytes.

    Attributes:
        name: Human-readable title.
        description: What the creator says the artifact depicts.
        content_hash: SHA-256 digest of the artifact bytes in this model.
        uri: Location used to retrieve the artifact in this model.
    """

    name: str
    description: str
    content_hash: str
    uri: str


class NotTheOwnerError(PermissionError):
    """Raised when someone tries to move another holder's token."""


class TokenDoesNotExistError(LookupError):
    """Raised when a token ID has not been minted in this collection."""

## 3. Build the collection: one token ID, one current owner

Here is the new abstraction, **defined here before we use it**. `NFTCollection` inherits the address and method dispatch from notebook 6's `SmartContract`.

Read the methods in this order:

1. `mint`: create the next individual ID and attach its owner and metadata. This collection chooses to let only its creator mint.
2. `owner_of` / `metadata_of`: read the registry entry.
3. `transfer`: require the current owner, then change that single entry.
4. `royalty_info` / `sell`: quote a creator's share and calculate a sale split.
5. `verify_content`: compare file bytes with the stored hash.

`royalty_bps=500` means **500 / 10,000 = 5%**. Basis points are percentages wearing a finance tie. `Decimal("2")` keeps the sale arithmetic exact without a detour into floating-point confetti.

**Scope:** this is an ERC-721-inspired Python model, with owner-only transfers. Full ERC-721 also has approvals, operators, safe transfers, and interface requirements. We leave those out so the individual ownership mapping stays visible. Our `sell` helper is extra classroom code; it returns proposed allocations and does not settle ETH. [ERC-721](https://eips.ethereum.org/EIPS/eip-721), [ERC-2981](https://eips.ethereum.org/EIPS/eip-2981).


In [3]:
class NFTCollection(SmartContract):
    """Track each token separately, with a creator and royalty quotation.

    Caller names stand in for authenticated wallet addresses. Ownership
    lives in Python memory; blocks record calls, as in notebook 6.
    """

    def __init__(self, address: str, creator: str, royalty_bps: int = 500) -> None:
        """Set the creator and rate; 500 basis points means five percent."""
        super().__init__(address)
        if not creator:
            raise ValueError("A collection needs a creator.")
        if not 0 <= royalty_bps <= 10_000:
            raise ValueError("Royalty basis points must be between 0 and 10,000.")
        self.creator = creator
        self.royalty_bps = royalty_bps
        self._next_id = 1
        self._owners: dict[int, str] = {}
        self._metadata: dict[int, TokenMetadata] = {}

    def mint(self, to: str, metadata: TokenMetadata, caller: str) -> int:
        """Create a token; this collection lets only its creator mint."""
        if caller != self.creator:
            raise PermissionError(f"Only {self.creator} can mint in this collection.")
        if not to:
            raise ValueError("A token needs a recipient.")
        token_id = self._next_id
        self._next_id += 1
        self._owners[token_id] = to
        self._metadata[token_id] = metadata
        return token_id

    def owner_of(self, token_id: int) -> str:
        """Return this token's owner, or reject an unminted ID."""
        if token_id not in self._owners:
            raise TokenDoesNotExistError(f"Token #{token_id} does not exist.")
        return self._owners[token_id]

    def metadata_of(self, token_id: int) -> TokenMetadata:
        """Read the pointer and hash attached to an existing token."""
        self.owner_of(token_id)
        return self._metadata[token_id]

    def transfer(self, token_id: int, caller: str, to: str) -> None:
        """Move a token only when its current owner is the caller."""
        current = self.owner_of(token_id)
        if caller != current:
            raise NotTheOwnerError(
                f"{caller} does not own token #{token_id}; {current} does."
            )
        if not to:
            raise ValueError("A transfer needs a recipient.")
        self._owners[token_id] = to

    def royalty_info(
        self, token_id: int, sale_price_eth: Decimal
    ) -> tuple[str, Decimal]:
        """Quote a royalty recipient and amount; no money moves here."""
        self.owner_of(token_id)
        if not sale_price_eth.is_finite() or sale_price_eth < 0:
            raise ValueError("Sale price must be finite and non-negative.")
        amount = sale_price_eth * Decimal(self.royalty_bps) / Decimal(10_000)
        return self.creator, amount

    def sell(
        self, token_id: int, caller: str, to: str, price_eth: Decimal
    ) -> dict[str, str | int | Decimal]:
        """Transfer a token and return its proposed payout split.

        This classroom helper computes allocations, not ETH settlement.
        It is separate from the basic transfer route, which has no price.
        """
        creator, royalty = self.royalty_info(token_id, price_eth)
        if price_eth == 0:
            raise ValueError("Use transfer for a gift; sale price must be positive.")
        # Validate the price before any ownership changes.
        self.transfer(token_id, caller, to)
        return {
            "token_id": token_id,
            "seller": caller,
            "buyer": to,
            "price_eth": price_eth,
            "seller_receives": price_eth - royalty,
            "creator": creator,
            "creator_royalty": royalty,
        }

    def verify_content(self, token_id: int, file_bytes: bytes) -> bool:
        """Check byte integrity, without deciding authorship or availability."""
        metadata = self.metadata_of(token_id)
        return hashlib.sha256(file_bytes).hexdigest() == metadata.content_hash

## 4. Prepare the artifact and deploy the contract

The document uses a photograph. We use a tiny text artifact as stand-in bytes, so this lesson runs without needing the actual photo. The mechanism is the same for a JPEG: hash its **bytes**, store the digest and pointer, keep the file separately.

`off_chain_files` will play the hosting service. It is a dictionary, because even our imaginary CDN is on a budget. The `.invalid` URI is deliberately fictional; nothing is downloaded or uploaded.

> **Pause and predict:** If we hash a file, have we stored the file on the chain? Could we reconstruct the file from those 64 hex characters?


In [4]:
artifact_name = "titus_fever_dream.txt"
artifact_bytes = (
    b"Titus wins the hackathon.\n"
    b"Voice: unavailable. Sinuses: staging a coup.\n"
    b"Five days ill. One questionable victory photo.\n"
)
artifact_hash = hashlib.sha256(artifact_bytes).hexdigest()
artifact_uri = "https://gallery.invalid/titus-fever-dream.txt"
off_chain_files = {artifact_uri: artifact_bytes}

metadata = TokenMetadata(
    name="Titus: Proof of Fever",
    description="Hackathon champion. Respiratory system runner-up.",
    content_hash=artifact_hash,
    uri=artifact_uri,
)

network = Network(["Node A", "Node B", "Node C"], random.Random(10))
chain = Blockchain([Validator(name, 100) for name in network.node_names])
collection = NFTCollection("titus.nft", creator="Titus", royalty_bps=500)

network.broadcast(
    Transaction("tx-deploy-nft", "Deploy NFTCollection at titus.nft"),
    origin="Node A",
)
_, deployment_block = network.include("Node A", "tx-deploy-nft", chain)

print(f"Off-chain artifact: {artifact_name} ({len(artifact_bytes)} bytes)")
print(f"SHA-256: {artifact_hash}")
print(f"Digest length: {len(artifact_hash)} hex characters")
print(f"Deployment receipt: Block #{deployment_block.index}")
print("The digest is a fingerprint. It is not a very compressed fever photo.")

Off-chain artifact: titus_fever_dream.txt (118 bytes)
SHA-256: e8d4773447ea2273972ded75b103dcc16349ce8b95fc5fee768aabf639f456dc
Digest length: 64 hex characters
Deployment receipt: Block #1
The digest is a fingerprint. It is not a very compressed fever photo.


## 5. Mint Titus. Finally, an asset with documented sinus risk.

Minting creates an entry in this collection. It does not create the file, check who took the photo, or decide the thing is worth 50 ETH.

We reuse `submit_call`: broadcast the call, include it in a block, then execute it. As in notebook 6, the Python object holds current state; the block holds a call receipt. The receipt includes the metadata representation, including the hash and URI. It does not contain `artifact_bytes`.

When someone says “token #1”, ask **which collection, on which chain?** Real identifiers need that context. Every collection can have a #1. The number is not a globally unique celebrity passport. [ERC-721 identifiers](https://eips.ethereum.org/EIPS/eip-721).


In [5]:
token_id, _, mint_block = submit_call(
    network,
    chain,
    "Node A",
    "tx-mint-titus",
    collection,
    "mint",
    to="Titus",
    metadata=metadata,
    caller="Titus",
)

print(f"Minted ({collection.address}, #{token_id}) -> {collection.owner_of(token_id)}")
print(f"Title: {collection.metadata_of(token_id).name}")
print(f"Pointer: {collection.metadata_of(token_id).uri}")
print(f"Royalty quote rate: {collection.royalty_bps / 100:.1f}%")
print(f"Block #{mint_block.index} records: {mint_block.data}")

Minted (titus.nft, #1) -> Titus
Title: Titus: Proof of Fever
Pointer: https://gallery.invalid/titus-fever-dream.txt
Royalty quote rate: 5.0%
Block #2 records: titus.nft.mint(to='Titus', metadata=TokenMetadata(name='Titus: Proof of Fever', description='Hackathon champion. Respiratory system runner-up.', content_hash='e8d4773447ea2273972ded75b103dcc16349ce8b95fc5fee768aabf639f456dc', uri='https://gallery.invalid/titus-fever-dream.txt'), caller='Titus')


## 6. Right-click save: excellent file acquisition, disappointing bank robbery

> **Pause and predict:** Jacob has identical bytes and an identical hash. Does `owner_of(1)` change? Can he transfer the token to himself?

The copy is real. The ownership update is a separate operation, with a separate rule. Possessing a file does not satisfy that rule.

Our names stand in for authenticated wallets. In a real contract, the caller comes from the transaction context. Here we pass it as a string so you can see the check; knowing how to type `caller="Titus"` is not real authentication.


In [6]:
jacob_copy = bytes(bytearray(artifact_bytes))
print(f"Identical bytes? {jacob_copy == artifact_bytes}")
print(
    f"Copy matches the token's hash? {collection.verify_content(token_id, jacob_copy)}"
)
print(f"Owner after right-click save: {collection.owner_of(token_id)}")

blocks_before_attempt = len(chain.chain)
try:
    submit_call(
        network,
        chain,
        "Node A",
        "tx-jacob-tries-theft",
        collection,
        "transfer",
        token_id=token_id,
        caller="Jacob",
        to="Jacob",
    )
except NotTheOwnerError as error:
    print(f"{type(error).__name__}: {error}")

print(f"Owner after rejected transfer: {collection.owner_of(token_id)}")
print(f"Attempt receipts added: {len(chain.chain) - blocks_before_attempt}")
print(
    "Included call != successful call. Notebook 7's receptionist is still not security."
)

Identical bytes? True
Copy matches the token's hash? True
Owner after right-click save: Titus
NotTheOwnerError: Jacob does not own token #1; Titus does.
Owner after rejected transfer: Titus
Attempt receipts added: 1
Included call != successful call. Notebook 7's receptionist is still not security.


## 7. Titus sells cheap. Bob discovers the phrase “price discovery”.

The document's two prices are **2 ETH**, then **50 ETH**. Keep them: they make the royalty arithmetic memorable.

At 5%, the first split is 1.9 ETH to the seller and 0.1 ETH to the creator. Titus is both, so his combined allocation is still 2 ETH. On Bob's 50 ETH resale, Bob's share is 47.5 ETH and Titus's creator share is 2.5 ETH.

The code changes token ownership and prints those allocations. **There is no ETH balance ledger here, so nobody has actually been paid.** A real marketplace must collect funds and settle the split. Calling a number `creator_royalty` does not make it arrive in a wallet, however confidently the variable is named.


In [7]:
first_sale, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-sale-to-bob",
    collection,
    "sell",
    token_id=token_id,
    caller="Titus",
    to="Bob",
    price_eth=Decimal("2"),
)
resale, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-bob-flips",
    collection,
    "sell",
    token_id=token_id,
    caller="Bob",
    to="Wei Yan",
    price_eth=Decimal("50"),
)

for label, sale in [("Titus -> Bob", first_sale), ("Bob -> Wei Yan", resale)]:
    print(f"{label}: price {sale['price_eth']} ETH")
    print(f"  Seller allocation: {sale['seller_receives']:.3f} ETH")
    print(
        f"  Creator allocation: {sale['creator_royalty']:.3f} ETH -> {sale['creator']}"
    )

print(f"Current owner: {collection.owner_of(token_id)}")
print("Payout arithmetic completed. Actual ETH settlement: outside this toy.")

Titus -> Bob: price 2 ETH
  Seller allocation: 1.900 ETH
  Creator allocation: 0.100 ETH -> Titus
Bob -> Wei Yan: price 50 ETH
  Seller allocation: 47.500 ETH
  Creator allocation: 2.500 ETH -> Titus
Current owner: Wei Yan
Payout arithmetic completed. Actual ETH settlement: outside this toy.


## 8. “Royalties forever” meets a transfer function with no price argument

The chapter gives royalties a rather enthusiastic sales pitch. Here is the catch in executable form.

ERC-2981 provides royalty payment information; it does **not** force payment. A wallet transfer could be a gift, a move between your own wallets, or the last step of a sale arranged elsewhere. The contract cannot infer a sale price from that transfer alone. [ERC-2981](https://eips.ethereum.org/EIPS/eip-2981).

Our `sell` helper computes the split. Our basic `transfer` does not ask about a sale. Watch Wei Yan use the latter. If a separate payment occurred elsewhere, this route would still not collect a royalty. “Automatic forever” now has a footnote large enough to rent out.

> **Pause and predict:** Will changing the owner through `transfer` call `royalty_info` or pay Titus anything?


In [8]:
receiver, quoted_royalty = collection.royalty_info(token_id, Decimal("50"))
print(f"At a 50 ETH price, the quote would be {quoted_royalty:.3f} ETH to {receiver}.")

submit_call(
    network,
    chain,
    "Node A",
    "tx-direct-transfer",
    collection,
    "transfer",
    token_id=token_id,
    caller="Wei Yan",
    to="Bob",
)

print(f"Owner after direct transfer: {collection.owner_of(token_id)}")
print("This path asked for no price and collected no royalty.")
print("A quote is information. A payment is an additional operation.")

At a 50 ETH price, the quote would be 2.500 ETH to Titus.
Owner after direct transfer: Bob
This path asked for no price and collected no royalty.
A quote is information. A payment is an additional operation.


## 9. Non-fungibility is individual tracking, not a JPEG cloning ban

Now mint a second token for a different artifact. Moving #1 did not move #2. These entries have individual identities and metadata.

Then mint a third token pointing at the **original file again**. Nothing in our mint rule requires a new content hash. Unique token IDs do not automatically mean unique artwork, a capped edition, or exclusive authorship. If a collection promises “only one”, examine the rule that actually prevents a second mint. The promise needs more than a dramatic font.


In [9]:
second_bytes = b"Titus after the hackathon: the trophy survives; the voice does not."
second_metadata = TokenMetadata(
    "Titus: The Sequel",
    "Same champion, even less voice.",
    hashlib.sha256(second_bytes).hexdigest(),
    "https://gallery.invalid/titus-sequel.txt",
)
off_chain_files[second_metadata.uri] = second_bytes
second_id, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-mint-sequel",
    collection,
    "mint",
    to="Titus",
    metadata=second_metadata,
    caller="Titus",
)
third_id, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-mint-same-file-again",
    collection,
    "mint",
    to="Titus",
    metadata=metadata,
    caller="Titus",
)

for existing_id in [token_id, second_id, third_id]:
    print(
        f"Token #{existing_id}: owner={collection.owner_of(existing_id)}, "
        f"title={collection.metadata_of(existing_id).name!r}"
    )
print(
    "Tokens #1 and #3 point at identical bytes?",
    collection.metadata_of(token_id).content_hash
    == collection.metadata_of(third_id).content_hash,
)
try:
    collection.owner_of(999)
except TokenDoesNotExistError as error:
    print(f"{type(error).__name__}: {error}")

Token #1: owner=Bob, title='Titus: Proof of Fever'
Token #2: owner=Titus, title='Titus: The Sequel'
Token #3: owner=Titus, title='Titus: Proof of Fever'
Tokens #1 and #3 point at identical bytes? True
TokenDoesNotExistError: Token #999 does not exist.


## 10. The counterfeit collection: the hash is honest about the wrong question

Jacob can create his own collection and mint a token pointing at the same bytes. This creates a **different token**, even if it is also numbered #1. He has copied neither the original collection address nor its ownership history.

A matching hash proves the bytes match a commitment. It does not prove the minter made the artifact, that an issuer is recognised, or that buyers will treat this collection as meaningful. Notebook 3's diploma problem has put on sunglasses and become a collectible.

A membership app should check the recognised collection and token. A check for “any token with this image” is a remarkably generous guest list.


In [10]:
imitation = NFTCollection("definitely-titus-trust-me.nft", creator="Jacob")
fake_id, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-imitation-mint",
    imitation,
    "mint",
    to="Jacob",
    metadata=metadata,
    caller="Jacob",
)

print(
    f"Recognised entry: ({collection.address}, #{token_id}) -> {collection.owner_of(token_id)}"
)
print(
    f"Imitation entry: ({imitation.address}, #{fake_id}) -> {imitation.owner_of(fake_id)}"
)
print(f"Imitation file hash matches? {imitation.verify_content(fake_id, jacob_copy)}")
print(f"Jacob owns the recognised token? {collection.owner_of(token_id) == 'Jacob'}")
print("The matching hash authenticated the bytes. It did not authenticate the artist.")

Recognised entry: (titus.nft, #1) -> Bob
Imitation entry: (definitely-titus-trust-me.nft, #1) -> Jacob
Imitation file hash matches? True
Jacob owns the recognised token? False
The matching hash authenticated the bytes. It did not authenticate the artist.


## 11. Integrity is useful. Immortality costs extra.

First, edit the file. Our verification notices the mismatch. This is what the digest is good at.

Second, remove the original from the hosting dictionary. We will still have an owner, a hash, a pointer, and a valid chain. We will not have a file at that location. Congratulations: a beautifully notarised missing attachment.

> **Pause and predict:** Does deleting the off-chain artifact add a block? Does `chain.is_valid()` start failing?


In [11]:
edited_bytes = artifact_bytes.replace(b"wins", b"politely attends")
print(f"Original matches? {collection.verify_content(token_id, artifact_bytes)}")
print(f"Edited version matches? {collection.verify_content(token_id, edited_bytes)}")

blocks_before_host_failure = len(chain.chain)
off_chain_files.pop(metadata.uri)

print(f"File available from our host? {metadata.uri in off_chain_files}")
print(f"Owner still recorded: {collection.owner_of(token_id)}")
print(f"URI still recorded: {collection.metadata_of(token_id).uri}")
print(f"Blocks added by host failure: {len(chain.chain) - blocks_before_host_failure}")
print(f"Chain check: {chain.is_valid()}")
print(
    f"Jacob's saved copy still verifies? {collection.verify_content(token_id, jacob_copy)}"
)
print("Right-click-save has unexpectedly become our disaster recovery department.")

Original matches? True
Edited version matches? False
File available from our host? False
Owner still recorded: Bob
URI still recorded: https://gallery.invalid/titus-fever-dream.txt
Blocks added by host failure: 0
Chain check: (True, 'Chain is valid.')
Jacob's saved copy still verifies? True
Right-click-save has unexpectedly become our disaster recovery department.


**Read the result.** A hash helps verify bytes you can obtain. It cannot recover missing bytes. The saved copy still matches, so losing one host does not erase every copy; it does break retrieval from that location.

IPFS addresses content rather than promising that someone will store it forever. Pinning protects content from garbage collection on participating nodes; continued availability still needs a provider holding and serving the data. Losing every provider can leave a valid identifier with nothing retrievable. [IPFS persistence and pinning](https://docs.ipfs.tech/concepts/persistence/).

This is notebook 8's two-book problem again. The chain records its side correctly. The host quietly stops doing its job. Cryptography passes; reality has left the meeting.


## 12. Why the auction headline tells you almost nothing about the typical token

Possible motivations include collecting, status, membership, access, and hoping the next buyer arrives with more money and fewer questions. The ownership mapping does not distinguish them. It also contains no price guarantee.

The chapter's broader point is about the distribution: a few spectacular winners can dominate the average. Below is a **made-up distribution**, not NFT market data and not a fitted power law. We give 990 items a zero value, nine a modest value, and one a ridiculous value to isolate that arithmetic.

Zero is a toy placeholder for no value in this example. In a real market, an unsold listing or an absent bid does not itself establish a completed sale price of zero.

> **Pause and predict:** Can the average exceed $1,000 while the median is $0?


In [12]:
toy_values_usd = [0] * 990 + [100] * 9 + [1_000_000]
print("ILLUSTRATIVE VALUES ONLY -- no live market data")
print(f"Items: {len(toy_values_usd):,}")
print(f"Mean: ${statistics.mean(toy_values_usd):,.2f}")
print(f"Median: ${statistics.median(toy_values_usd):,.2f}")
print(f"Largest: ${max(toy_values_usd):,.0f}")
print("Headline: million-dollar collectible. Typical item in our example: zero.")
print("The average has hired the biggest number as its publicist.")

ILLUSTRATIVE VALUES ONLY -- no live market data
Items: 1,000
Mean: $1,000.90
Median: $0.00
Largest: $1,000,000
Headline: million-dollar collectible. Typical item in our example: zero.
The average has hired the biggest number as its publicist.


## 13. Useful mechanism, optional circus

| Use | What the token can track | What the surrounding system must supply |
| --- | --- | --- |
| Ticket | Current holder of a particular admission token | An issuer and a door that honours it; reuse rules |
| Membership | Holder of a recognised collection's token | Access checks and benefits someone actually provides |
| Game item | Holder of an individually identified item | A game that reads and respects that record |
| Credential | An issuer's individually identified record | Trust in the issuer, privacy and revocation design; often transfer restrictions |
| Domain | A registry entry associated with a holder | Naming and resolution rules |
| Collectible | Identity, metadata and transfer history | Recognition, storage, and a willing buyer |

Owning this toy token updates a registry entry. We have written no licence terms, copyright assignment, or event admission agreement into the model. Those are additional arrangements; the file hash cannot invent them.

The NFT is a mechanism. The party organised around it is a separate product, and sometimes that product is a Discord server with three moderators and a roadmap drawn in crayon.


## What this toy leaves out

This is Python, not deployed Solidity or a complete ERC-721 implementation. Caller strings are a classroom stand-in for wallet authentication. Minting is creator-only by our choice; metadata is fixed by our public methods, and our mint rule does not cap the edition or ban duplicate hashes.

State stays in Python objects. Notebook 6's `submit_call` records a call before running it, so a rejected call can leave an attempt receipt. We do not implement consensus execution, state replay, general rollback, gas, approvals, safe receiver callbacks, burns, or real payment settlement. Price validation happens before transfers in our sale helper.

File availability comes from the toy hosting dictionary. The real-world issuer, content storage, access application, and any sale settlement need their own rules. The interesting boundary is visible even with the machinery stripped down: **record integrity does not supply external truth or continued service**.


## Revision checkpoint: close the code and explain these

1. Jacob has the exact bytes. Why can he not transfer Titus's token?
2. What identifies token #1 completely? Why is the number alone insufficient?
3. Can tokens #1 and #3 point at the same file? What rule would limit that?
4. What did the two sales actually change in our model? What did they merely calculate?
5. Why does a direct transfer bypass our royalty calculation?
6. Why does the chain stay valid when the host loses the artifact?
7. Which fact does a matching content hash prove, and which facts does it leave unanswered?

<details>
<summary>Answers, for when future Titus is also running a fever</summary>

1. The transfer rule checks the token's current owner. File possession is separate.
2. The chain, collection address and token ID give the context; another collection can also mint #1.
3. Yes. Our mint rule checks the creator and recipient, not unique hashes or a supply cap.
4. They changed ownership and calculated payout allocations. No ETH balances moved.
5. It supplies no price and invokes no royalty quote or payment step.
6. Off-chain deletion changes neither block payloads nor hash links.
7. It proves the provided bytes match the stored digest. It does not establish authorship, issuer recognition, value, or future availability.

</details>

**Carry this into the earlier notebooks:** the Merkle proof commits to data, the oracle supplies a price, the issuer records reserves, and the NFT points at an artifact. For each, ask who supplies the outside fact and what happens when they stop. The blockchain can keep immaculate minutes of a meeting whose entire premise was nonsense.


## Sources and relationship to the document

Adapted from **“Chapter 5: In Which Titus Is Minted”** and the closing reflections in your `blockchain_303.docx`. The feverish hackathon subject, cast, 2 ETH / 50 ETH examples, and mechanism-first approach come from that chapter. The toy distribution is newly constructed for revision.

The notebook narrows the chapter's blanket claims about off-chain files, unique artwork, and perpetual royalty enforcement. It demonstrates the actual separation between token ownership, file integrity, hosting, and payment.

- [ERC-721: Non-Fungible Token Standard](https://eips.ethereum.org/EIPS/eip-721) — individual token ownership, transfers, collection-scoped identifiers, and optional metadata.
- [ERC-2981: NFT Royalty Standard](https://eips.ethereum.org/EIPS/eip-2981) — royalty payment information; payment is separate and voluntary under this standard.
- [IPFS: Persistence, permanence, and pinning](https://docs.ipfs.tech/concepts/persistence/) — content addressing and the work needed to keep content available.

The new types are defined inline above. Their reusable copies live in [`blockchain_lib/nft.py`](../blockchain_lib/nft.py), so a later lesson can import them after you have seen how they work.
